# NB_SEC_Sync · Phase 4 · metamodel → Entra groups, workspace roles and OneLake roles
Desired-state sync. For every secured lakehouse it sends the **complete** role set (SEC-03), validates with `dryRun=true`,
applies with `If-Match` (SEC-04), stops on drift (POL-09) and writes SyncRun / AuditLog / RoleBinding evidence.

In [ ]:
DRY_RUN = False        # True: compute and validate only, change nothing
FORCE_ITEMS = ""       # comma-separated lakehouse names to overwrite even if drift is detected
TRIGGER = "OnDemand"   # Scheduled | OnDemand

In [ ]:
%run NB_SEC_Common

In [ ]:
run_id = int(sql_exec("SELECT ISNULL(MAX(RunId),0)+1 FROM sec.SyncRun", fetch_scalar=True))
started = NOW()
if not DRY_RUN:
    sql_exec("INSERT INTO sec.SyncRun (RunId,StartedAt,TriggerType,DryRun,RolesChanged,MembersAdded,MembersRemoved,Errors) VALUES (?,?,?,?,0,0,0,0)",
             (run_id, started.isoformat(sep=" "), TRIGGER, False))
stats = {"roles": 0, "added": 0, "removed": 0, "errors": 0}
FAILURES = []
def log(action, target, request_id=None, before=None, after=None, outcome="Success"):
    print(f"[{outcome}] {action} {target}")
    if outcome != "Success":
        stats["errors"] += 1
        FAILURES.append(f"{action} {target}: {json.dumps(after)[:400] if after else ''}")
    if not DRY_RUN: audit(action, target, run_id, request_id, before, after, outcome)

In [ ]:
# 1. Load the metamodel
users = sql_df("SELECT UserId, UPN, EntraObjectId, EmploymentStatus FROM sec.UserProfile")
reqs = sql_df("SELECT RequestId, UserId, RoleTemplateId, ScopeValue, Status FROM sec.AccessRequest WHERE Status IN ('Active','Approved')")
tpls = sql_df("SELECT RoleTemplateId, RoleCode, ScopeType, WorkspaceRole FROM sec.RoleTemplate WHERE IsActive = 1")
objr = sql_df("SELECT RoleTemplateId, ItemKey, TablePath FROM sec.ObjectRule")
rowr = sql_df("SELECT RoleTemplateId, ItemKey, TablePath, FilterColumn FROM sec.RowRule")
colr = sql_df("SELECT RoleTemplateId, ItemKey, TablePath, AllowedColumns FROM sec.ColumnRule")
items = sql_df("SELECT i.ItemKey, i.ItemName, i.ItemType, i.OneLakeSecured, w.WorkspaceName FROM sec.Item i JOIN sec.Workspace w ON w.WorkspaceKey = i.WorkspaceKey")
prev = sql_df("SELECT RoleName, ItemKey, RoleJson, Status FROM sec.RoleBinding")

# 2. Resolve Fabric IDs by name (no GUIDs need to be typed into the metamodel)
ws_id = {w["displayName"]: w["id"] for w in fabric_all("/workspaces")}
item_id = {}
for it in items.itertuples():
    wid = ws_id.get(it.WorkspaceName)
    if not wid: log("RESOLVE_FAILED", f"workspace {it.WorkspaceName}", outcome="Failure"); continue
    for x in fabric_all(f"/workspaces/{wid}/items"):
        if x["displayName"] == it.ItemName: item_id[it.ItemKey] = (wid, x["id"])

In [ ]:
# 3. Desired bindings: one per (template, scope); members are users with Active requests
code = dict(zip(tpls.RoleTemplateId, tpls.RoleCode))
def role_name(tid, scope): return code[tid] + ("All" if scope == "ALL" else "OU" + str(scope))
active_users = set(users[users.EmploymentStatus == "Active"].UserId)
bindings = {}
for q in reqs.itertuples():
    if q.RoleTemplateId not in code: continue
    b = bindings.setdefault(role_name(q.RoleTemplateId, q.ScopeValue), {"tid": q.RoleTemplateId, "scope": q.ScopeValue, "users": set(), "req": {}})
    if q.Status == "Active" and q.UserId in active_users:
        b["users"].add(q.UserId); b["req"][q.UserId] = int(q.RequestId)

# 4. Entra: resolve users, ensure one group per binding, reconcile members
upn = dict(zip(users.UserId, users.UPN))
oid = {}
for u in users.itertuples():
    if u.EmploymentStatus != "Active": continue
    r = call("graph", "GET", f"/users/{u.UPN}?$select=id", ok=None)
    if r.status_code == 200:
        oid[u.UserId] = r.json()["id"]
        if not DRY_RUN and u.EntraObjectId is None:
            sql_exec("UPDATE sec.UserProfile SET EntraObjectId = ?, ExistsInTenant = 1 WHERE UserId = ?", (oid[u.UserId], int(u.UserId)))
def graph_retry(method, url, **kw):
    # Entra is eventually consistent: a group created seconds ago can 404 for a short while
    for attempt in range(8):
        r = call("graph", method, url, ok=None, **kw)
        if r.status_code != 404: break
        time.sleep(3 + 2 * attempt)
    if r.status_code not in (200, 201, 204):
        raise RuntimeError(f"{method} {url} -> {r.status_code}: {r.text[:500]}")
    return r

for name, b in bindings.items():
    g = f"grp-ol-{name}"
    created = False
    found = graph_all(f"/groups?$filter=displayName eq '{g}'&$select=id")
    if found:
        gid = found[0]["id"]
    elif DRY_RUN:
        gid = None; print("would create group", g)
    else:
        gid = call("graph", "POST", "/groups", json={"displayName": g, "mailEnabled": False, "mailNickname": g.replace("-", ""), "securityEnabled": True,
                                                     "description": "OneLake role group. Managed by NB_SEC_Sync; do not edit by hand."}).json()["id"]
        log("GROUP_CREATE", g, after={"id": gid}); created = True
    b["gid"] = gid
    current = {m["id"] for m in graph_all(f"/groups/{gid}/members?$select=id")} if gid and not created else set()
    want = {oid[u] for u in b["users"] if u in oid}
    for u in b["users"] - set(oid):
        log("USER_NOT_FOUND", f"{upn[u]} for {g}", request_id=b["req"].get(u), outcome="Failure")
    for m in want - current:
        if not DRY_RUN:
            graph_retry("POST", f"/groups/{gid}/members/$ref", json={"@odata.id": f"{GRAPH}/directoryObjects/{m}"})
        stats["added"] += 1
        u = next(k for k, v in oid.items() if v == m)
        log("GROUP_MEMBER_ADD", g, request_id=b["req"].get(u), after={"member": upn[u]})
    for m in current - want:
        if not DRY_RUN:
            call("graph", "DELETE", f"/groups/{gid}/members/{m}/$ref")
        stats["removed"] += 1
        log("GROUP_MEMBER_REMOVE", g, before={"memberObjectId": m})

In [ ]:
# 5. Compile roles per secured lakehouse
def tpath(p): return p.replace("/Tables/", "").replace("/", ".")
def compile_role(name, b, item_key):
    tid, scope = b["tid"], b["scope"]
    paths = sorted(objr[(objr.RoleTemplateId == tid) & (objr.ItemKey == item_key)].TablePath)
    if not paths: return None
    rule = {"effect": "Permit", "permission": [{"attributeName": "Path", "attributeValueIncludedIn": paths},
                                               {"attributeName": "Action", "attributeValueIncludedIn": ["Read"]}]}
    cons = {}
    rr = rowr[(rowr.RoleTemplateId == tid) & (rowr.ItemKey == item_key)]
    if len(rr) and scope != "ALL":
        cons["rows"] = [{"tablePath": x.TablePath, "value": f"SELECT * FROM {tpath(x.TablePath)} WHERE {x.FilterColumn} = {int(scope)}"} for x in rr.itertuples()]
    cr = colr[(colr.RoleTemplateId == tid) & (colr.ItemKey == item_key)]
    if len(cr):
        cons["columns"] = [{"tablePath": x.TablePath, "columnNames": json.loads(x.AllowedColumns), "columnEffect": "Permit", "columnAction": ["Read"]} for x in cr.itertuples()]
    if cons: rule["constraints"] = cons
    members = [{"tenantId": CONFIG["tenant_id"], "objectId": b["gid"], "objectType": "Group"}] if b.get("gid") else []
    return {"name": name, "decisionRules": [rule], "members": {"microsoftEntraMembers": members}}

def _canon_role(r):
    # Compare only what we govern; Fabric echoes roles back with extra/defaulted fields and its own ordering
    def low(x): return str(x).strip().lower()
    rules = []
    for rule in r.get("decisionRules", []):
        perm = {low(p.get("attributeName")): sorted(low(v) for v in p.get("attributeValueIncludedIn", [])) for p in rule.get("permission", [])}
        cons = rule.get("constraints") or {}
        cols = sorted((low(c.get("tablePath")), tuple(sorted(low(n) for n in c.get("columnNames", [])))) for c in cons.get("columns") or [])
        rows = sorted((low(x.get("tablePath")), " ".join(low(x.get("value")).split())) for x in cons.get("rows") or [])
        rules.append({"effect": low(rule.get("effect", "permit")), "perm": perm, "cols": cols, "rows": rows})
    mem = sorted(low(m.get("objectId")) for m in (r.get("members") or {}).get("microsoftEntraMembers") or [])
    return {"name": low(r.get("name")), "rules": sorted(rules, key=lambda x: json.dumps(x, sort_keys=True)), "members": mem}

def norm(roles):
    return json.dumps(sorted((_canon_role(r) for r in roles), key=lambda x: x["name"]), sort_keys=True)

def first_diff(a_roles, b_roles):
    a = {x["name"]: x for x in json.loads(norm(a_roles))}; b = {x["name"]: x for x in json.loads(norm(b_roles))}
    for n in sorted(set(a) | set(b)):
        if a.get(n) != b.get(n): return {"role": n, "live": a.get(n), "expected": b.get(n)}
    return None

force = {x.strip() for x in FORCE_ITEMS.split(",") if x.strip()}
new_bindings = []
for it in items[items.OneLakeSecured == 1].itertuples():
    if it.ItemKey not in item_id: log("RESOLVE_FAILED", it.ItemName, outcome="Failure"); continue
    wid, iid = item_id[it.ItemKey]
    desired = []
    for name, b in sorted(bindings.items()):
        role = compile_role(name, b, it.ItemKey)
        if role: desired.append(role); new_bindings.append((name, it.ItemKey, b, role))
    try:
        live = call("fabric", "GET", f"/workspaces/{wid}/items/{iid}/dataAccessRoles")
    except RuntimeError as e:
        log("ROLE_READ_FAILED", it.ItemName, after={"error": str(e)[:800]}, outcome="Failure"); continue
    etag = live.headers.get("ETag"); live_roles = live.json().get("value", [])
    last = [json.loads(x) for x in prev[(prev.ItemKey == it.ItemKey) & (prev.Status == "Applied")].RoleJson]
    if norm(live_roles) == norm(desired):
        print(f"{it.ItemName}: no change"); continue
    if last and norm(live_roles) != norm(last) and it.ItemName not in force:
        log("DRIFT_DETECTED", it.ItemName, before={"lastApplied": [r["name"] for r in last]}, after={"live": [r["name"] for r in live_roles], "diff": first_diff(live_roles, last)}, outcome="Failure")
        continue
    body = {"value": desired}
    try:
        call("fabric", "PUT", f"/workspaces/{wid}/items/{iid}/dataAccessRoles?dryRun=true", json=body)
        if not DRY_RUN:
            call("fabric", "PUT", f"/workspaces/{wid}/items/{iid}/dataAccessRoles", json=body, headers={"If-Match": etag} if etag else {})
        stats["roles"] += 1
        log("ROLE_UPSERT" if not DRY_RUN else "DRY_RUN", it.ItemName, before={"roles": sorted(r["name"] for r in live_roles)}, after={"roles": sorted(r["name"] for r in desired)})
    except RuntimeError as e:
        log("ROLE_APPLY_FAILED", it.ItemName, after={"error": str(e)[:800]}, outcome="Failure")

In [ ]:
# 6. Workspace RBAC: each binding's group is Viewer on the workspaces of its lakehouses (IAM-05)
for wid in {item_id[k][0] for (_, k, _, _) in new_bindings if k in item_id}:
    have = {(a["principal"]["id"]) for a in fabric_all(f"/workspaces/{wid}/roleAssignments")}
    for gid, gname in {(b.get("gid"), f"grp-ol-{n}") for (n, k, b, _) in new_bindings if k in item_id and item_id[k][0] == wid}:
        if gid and gid not in have:
            if not DRY_RUN:
                call("fabric", "POST", f"/workspaces/{wid}/roleAssignments", json={"principal": {"id": gid, "type": "Group"}, "role": "Viewer"})
            log("WORKSPACE_ROLE_ADD", gname, after={"workspaceId": wid, "role": "Viewer"})

In [ ]:
# 6b. Retired bindings: a role that no longer has any Active or Approved request (for example an expired audit grant).
#     The OneLake role is already gone (step 5 sends the complete set); here the group is emptied and loses its
#     workspace Viewer role, so the person no longer even sees the workspace.
retired = sorted(set(prev.RoleName) - set(bindings))
for name in retired:
    g = f"grp-ol-{name}"
    found = graph_all(f"/groups?$filter=displayName eq '{g}'&$select=id")
    if not found: continue
    gid = found[0]["id"]
    for m in graph_all(f"/groups/{gid}/members?$select=id"):
        if not DRY_RUN:
            call("graph", "DELETE", f"/groups/{gid}/members/{m['id']}/$ref")
        stats["removed"] += 1
        log("GROUP_MEMBER_REMOVE", g, before={"memberObjectId": m["id"]}, after={"reason": "role retired"})
    for k in set(prev[prev.RoleName == name].ItemKey):
        if k not in item_id: continue
        wid = item_id[k][0]
        for a in fabric_all(f"/workspaces/{wid}/roleAssignments"):
            if a["principal"]["id"] == gid:
                if not DRY_RUN:
                    call("fabric", "DELETE", f"/workspaces/{wid}/roleAssignments/{a['id']}")
                log("WORKSPACE_ROLE_REMOVE", g, before={"workspaceId": wid, "role": a.get("role")})


In [ ]:
# 7. Evidence: RoleBinding desired state, SyncRun totals
if not DRY_RUN:
    sql_exec("UPDATE sec.RoleBinding SET Status = 'Removed', MemberCount = 0")
    for name, item_key, b, role in new_bindings:
        rj = json.dumps(role, separators=(",", ":"))
        sql_exec("""MERGE sec.RoleBinding AS t USING (SELECT ? RoleName, ? ItemKey) s ON t.RoleName = s.RoleName AND t.ItemKey = s.ItemKey
                    WHEN MATCHED THEN UPDATE SET RoleTemplateId = ?, ScopeValue = ?, EntraGroupName = ?, EntraGroupId = ?, MemberCount = ?, RoleJson = ?,
                         Status = ?, AppliedAt = SYSUTCDATETIME()
                    WHEN NOT MATCHED THEN INSERT (RoleName,ItemKey,RoleTemplateId,ScopeValue,EntraGroupName,EntraGroupId,MemberCount,RoleJson,Status,AppliedAt)
                         VALUES (?,?,?,?,?,?,?,?,?,SYSUTCDATETIME());""",
                 (name, int(item_key), int(b["tid"]), b["scope"], f"grp-ol-{name}", b.get("gid"), len(b["users"]), rj, "Applied" if b["users"] else "Planned",
                  name, int(item_key), int(b["tid"]), b["scope"], f"grp-ol-{name}", b.get("gid"), len(b["users"]), rj, "Applied" if b["users"] else "Planned"))
    sql_exec("UPDATE sec.SyncRun SET EndedAt = ?, RolesChanged = ?, MembersAdded = ?, MembersRemoved = ?, Errors = ? WHERE RunId = ?",
             (NOW().isoformat(sep=" "), stats["roles"], stats["added"], stats["removed"], stats["errors"], run_id))
print(f"Run {run_id} {'(dry run) ' if DRY_RUN else ''}finished:", stats)
for f in FAILURES: print("FAILED:", f)
if stats["errors"]:
    raise Exception(f"Sync finished with {stats['errors']} errors; see sec.AuditLog for RunId {run_id}")   # fails the pipeline so the security team is alerted (NFR-04)